In [243]:
# pip install openpyxl

In [244]:
import pandas as pd
import openpyxl
import os

In [245]:
annotation_file = 'data/hazard-project-annotation.xlsb.xlsm'

Extract URLs

In [246]:

workbook = openpyxl.load_workbook(annotation_file)
sheet = workbook.active  

image_link_column = None
for col in sheet.iter_cols(1, sheet.max_column):
    if col[0].value == "Image Link":  
        image_link_column = col[0].column  
        break

        
image_links = []

for row in range(2, sheet.max_row + 1):  
    cell = sheet.cell(row=row, column=image_link_column)

    if cell.value and isinstance(cell.value, str) and "HYPERLINK" in cell.value:
        start_index = cell.value.find('"') + 1
        end_index = cell.value.find('"', start_index)
        if start_index != -1 and end_index != -1:
            hyperlink_url = cell.value[start_index:end_index] 
            image_links.append(hyperlink_url)  

    elif cell.hyperlink:
        image_links.append(cell.hyperlink.target)  
        
    else:
        print(f"Row {row} does not have a hyperlink.")

print(f"Total hyperlinks extracted: {len(image_links)}")


Total hyperlinks extracted: 2615


In [247]:
df = pd.read_excel(annotation_file)
df['Image Link'] = image_links

In [248]:
df['File'].value_counts()/5

File
Road_Condition_1         60.0
Driving_Distraction_1    58.0
Pedestrian_Crossing_1    54.0
Driving_Distraction_0    51.0
Pedestrian_Crossing_0    51.0
Vehicle_Load_0           51.0
Traffic_Rules_1          50.0
Vehicle_Load_1           50.0
Road_Condition_0         49.0
Traffic_Rules_0          49.0
Name: count, dtype: float64

Fix Wrong File Error

In [249]:
df[df['Image Link'].str.contains('highwaycode')]

,Domain,Image Link,Label,Rule,File
2104,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Not Applicable,Driving Distraction,Traffic_Rules_0
2105,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Not Applicable,Pedestrian Crossing,Traffic_Rules_0
2106,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Complied,Road Condition,Traffic_Rules_0
2107,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Violated,Traffic Rules,Traffic_Rules_0
2108,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Complied,Vehicle Load,Traffic_Rules_0


In [250]:
df[2100:2110]

,Domain,Image Link,Label,Rule,File
2100,Traffic,https://media.istockphoto.com/id/1219457616/ph...,Not Applicable,Driving Distraction,Traffic_Rules_0
2101,Traffic,https://media.istockphoto.com/id/1219457616/ph...,Not Applicable,Pedestrian Crossing,Traffic_Rules_0
2102,Traffic,https://media.istockphoto.com/id/1219457616/ph...,Not Applicable,Road Condition,Traffic_Rules_0
2103,Traffic,https://media.istockphoto.com/id/1219457616/ph...,Violated,Traffic Rules,Traffic_Rules_0
2104,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Not Applicable,Driving Distraction,Traffic_Rules_0
2105,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Not Applicable,Pedestrian Crossing,Traffic_Rules_0
2106,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Complied,Road Condition,Traffic_Rules_0
2107,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Violated,Traffic Rules,Traffic_Rules_0
2108,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Complied,Vehicle Load,Traffic_Rules_0
2109,Traffic,https://media.istockphoto.com/id/1219457616/ph...,Complied,Vehicle Load,Traffic_Rules_0


In [251]:
# File error
df.loc[df['Image Link'] == 'https://media.istockphoto.com/id/1219457616/photo/hwy-sy-west-dist-2-bridge-jam.jpg?s=1024x1024&w=is&k=20&c=i-gmb0ZHC6UiaJPTlm_wXVYd3nBsHqKevm8Jyhbex-o=', 'File'] = 'Traffic_Rules_1'
df.loc[df['Image Link'] == 'https://www.highwaycode.com.ng/uploads/3/2/9/2/3292309/6877789.png?504', 'File'] = 'Traffic_Rules_1'

row_to_move = df.loc[2109]
df = df.drop(index=2109)
top = df.iloc[:2104]
bottom = df.iloc[2104:]
df = pd.concat([top, pd.DataFrame([row_to_move]), bottom]).reset_index(drop=True)


In [252]:
df[2100:2110]

,Domain,Image Link,Label,Rule,File
2100,Traffic,https://media.istockphoto.com/id/1219457616/ph...,Not Applicable,Driving Distraction,Traffic_Rules_1
2101,Traffic,https://media.istockphoto.com/id/1219457616/ph...,Not Applicable,Pedestrian Crossing,Traffic_Rules_1
2102,Traffic,https://media.istockphoto.com/id/1219457616/ph...,Not Applicable,Road Condition,Traffic_Rules_1
2103,Traffic,https://media.istockphoto.com/id/1219457616/ph...,Violated,Traffic Rules,Traffic_Rules_1
2104,Traffic,https://media.istockphoto.com/id/1219457616/ph...,Complied,Vehicle Load,Traffic_Rules_1
2105,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Not Applicable,Driving Distraction,Traffic_Rules_1
2106,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Not Applicable,Pedestrian Crossing,Traffic_Rules_1
2107,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Complied,Road Condition,Traffic_Rules_1
2108,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Violated,Traffic Rules,Traffic_Rules_1
2109,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Complied,Vehicle Load,Traffic_Rules_1


In [253]:
df['Domain'] = df['Domain'].astype(str).str.strip()
df['File'] = df['File'].astype(str).str.strip()
df['Rule'] = df['Rule'].astype(str).str.strip()

df['Image ID'] = (
    df.groupby(['Domain', 'File', 'Rule'])
    .cumcount() + 1
).apply(lambda x: f"{x:07d}")
df

,Domain,Image Link,Label,Rule,File,Image ID
0,Traffic,https://d18hvbbehx7xaf.cloudfront.net/public/u...,Complied,Driving Distraction,Driving_Distraction_0,0000001
1,Traffic,https://d18hvbbehx7xaf.cloudfront.net/public/u...,Not Applicable,Pedestrian Crossing,Driving_Distraction_0,0000001
2,Traffic,https://d18hvbbehx7xaf.cloudfront.net/public/u...,Not Applicable,Road Condition,Driving_Distraction_0,0000001
3,Traffic,https://d18hvbbehx7xaf.cloudfront.net/public/u...,Not Applicable,Traffic Rules,Driving_Distraction_0,0000001
4,Traffic,https://d18hvbbehx7xaf.cloudfront.net/public/u...,Not Applicable,Vehicle Load,Driving_Distraction_0,0000001
...,...,...,...,...,...,...
2610,Traffic,https://encrypted-tbn0.gstatic.com/images?q=tb...,Not Applicable,Driving Distraction,Vehicle_Load_1,0000050
2611,Traffic,https://encrypted-tbn0.gstatic.com/images?q=tb...,Not Applicable,Pedestrian Crossing,Vehicle_Load_1,0000050
2612,Traffic,https://encrypted-tbn0.gstatic.com/images?q=tb...,Not Applicable,Road Condition,Vehicle_Load_1,0000050
2613,Traffic,https://encrypted-tbn0.gstatic.com/images?q=tb...,Not Applicable,Traffic Rules,Vehicle_Load_1,0000050


In [254]:
df[df['Image Link'] == 'https://www.highwaycode.com.ng/uploads/3/2/9/2/3292309/6877789.png?504']

,Domain,Image Link,Label,Rule,File,Image ID
2105,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Not Applicable,Driving Distraction,Traffic_Rules_1,0000052
2106,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Not Applicable,Pedestrian Crossing,Traffic_Rules_1,0000052
2107,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Complied,Road Condition,Traffic_Rules_1,0000052
2108,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Violated,Traffic Rules,Traffic_Rules_1,0000052
2109,Traffic,https://www.highwaycode.com.ng/uploads/3/2/9/2...,Complied,Vehicle Load,Traffic_Rules_1,0000052


Sync with Processed Images

In [255]:
base_dir='data/images_link'
processed_dir = 'data/processed_images'

url_ids = dict()

for domain in os.listdir(base_dir):
    domain_path = os.path.join(base_dir, domain)
    if not os.path.isdir(domain_path):
        continue
    
    for txt_file in os.listdir(domain_path):
        
        if not txt_file.endswith('.txt'):
            continue

        label = os.path.splitext(txt_file)[0]
        rule_folder = label
        txt_path = os.path.join(domain_path, txt_file)

        with open(txt_path, 'r') as f:
            links = [line.strip() for line in f if line.strip()]

        for i, url in enumerate(links):
            image_id = f"{i+1:07d}"
            full_path = None
            for ext in ['.jpg', '.jpeg', '.png']:
                candidate = os.path.join(processed_dir, domain, rule_folder, image_id + ext)
                if os.path.exists(candidate):
                    full_path = candidate
                    break

            uniq_key = f"{domain}/{label}::{url}"
            if full_path and uniq_key not in url_ids:
                url_ids[uniq_key] = full_path


In [256]:
df['Image Path'] = (df['Domain'].apply(lambda x: x.lower())+'/'+df['File']+'::'+df['Image Link']).map(url_ids)
df

,Domain,Image Link,Label,Rule,File,Image ID,Image Path
0,Traffic,https://d18hvbbehx7xaf.cloudfront.net/public/u...,Complied,Driving Distraction,Driving_Distraction_0,0000001,data/processed_images/traffic/Driving_Distract...
1,Traffic,https://d18hvbbehx7xaf.cloudfront.net/public/u...,Not Applicable,Pedestrian Crossing,Driving_Distraction_0,0000001,data/processed_images/traffic/Driving_Distract...
2,Traffic,https://d18hvbbehx7xaf.cloudfront.net/public/u...,Not Applicable,Road Condition,Driving_Distraction_0,0000001,data/processed_images/traffic/Driving_Distract...
3,Traffic,https://d18hvbbehx7xaf.cloudfront.net/public/u...,Not Applicable,Traffic Rules,Driving_Distraction_0,0000001,data/processed_images/traffic/Driving_Distract...
4,Traffic,https://d18hvbbehx7xaf.cloudfront.net/public/u...,Not Applicable,Vehicle Load,Driving_Distraction_0,0000001,data/processed_images/traffic/Driving_Distract...
...,...,...,...,...,...,...,...
2610,Traffic,https://encrypted-tbn0.gstatic.com/images?q=tb...,Not Applicable,Driving Distraction,Vehicle_Load_1,0000050,data/processed_images/traffic/Vehicle_Load_1/0...
2611,Traffic,https://encrypted-tbn0.gstatic.com/images?q=tb...,Not Applicable,Pedestrian Crossing,Vehicle_Load_1,0000050,data/processed_images/traffic/Vehicle_Load_1/0...
2612,Traffic,https://encrypted-tbn0.gstatic.com/images?q=tb...,Not Applicable,Road Condition,Vehicle_Load_1,0000050,data/processed_images/traffic/Vehicle_Load_1/0...
2613,Traffic,https://encrypted-tbn0.gstatic.com/images?q=tb...,Not Applicable,Traffic Rules,Vehicle_Load_1,0000050,data/processed_images/traffic/Vehicle_Load_1/0...


Remove Duplicates and Format Error

In [257]:
df['Image ID_'] = df['Image Path'].apply(lambda x:str(x).split('/')[-1].split('.')[0])

# duplication
len(df[(df['Image ID'] != df['Image ID_']) & (~df['Image Path'].isna())])/5

6.0

In [258]:
df[(df['Image ID'] != df['Image ID_']) & (~df['Image Path'].isna())]

,Domain,Image Link,Label,Rule,File,Image ID,Image Path,Image ID_
185,Traffic,https://media.istockphoto.com/id/2166802770/ph...,Complied,Driving Distraction,Driving_Distraction_0,0000038,data/processed_images/traffic/Driving_Distract...,0000028
186,Traffic,https://media.istockphoto.com/id/2166802770/ph...,Not Applicable,Pedestrian Crossing,Driving_Distraction_0,0000038,data/processed_images/traffic/Driving_Distract...,0000028
187,Traffic,https://media.istockphoto.com/id/2166802770/ph...,Not Applicable,Road Condition,Driving_Distraction_0,0000038,data/processed_images/traffic/Driving_Distract...,0000028
188,Traffic,https://media.istockphoto.com/id/2166802770/ph...,Not Applicable,Traffic Rules,Driving_Distraction_0,0000038,data/processed_images/traffic/Driving_Distract...,0000028
189,Traffic,https://media.istockphoto.com/id/2166802770/ph...,Not Applicable,Vehicle Load,Driving_Distraction_0,0000038,data/processed_images/traffic/Driving_Distract...,0000028
730,Traffic,https://media.istockphoto.com/id/478525372/pho...,Not Applicable,Driving Distraction,Pedestrian_Crossing_0,0000038,data/processed_images/traffic/Pedestrian_Cross...,0000018
731,Traffic,https://media.istockphoto.com/id/478525372/pho...,Complied,Pedestrian Crossing,Pedestrian_Crossing_0,0000038,data/processed_images/traffic/Pedestrian_Cross...,0000018
732,Traffic,https://media.istockphoto.com/id/478525372/pho...,Complied,Road Condition,Pedestrian_Crossing_0,0000038,data/processed_images/traffic/Pedestrian_Cross...,0000018
733,Traffic,https://media.istockphoto.com/id/478525372/pho...,Complied,Traffic Rules,Pedestrian_Crossing_0,0000038,data/processed_images/traffic/Pedestrian_Cross...,0000018
734,Traffic,https://media.istockphoto.com/id/478525372/pho...,Complied,Vehicle Load,Pedestrian_Crossing_0,0000038,data/processed_images/traffic/Pedestrian_Cross...,0000018


In [259]:
# format (.webp / link not found error)
len(df[df['Image Path'].isna()])/5

3.0

In [260]:
df[(df['Image Path'].isna())]

,Domain,Image Link,Label,Rule,File,Image ID,Image Path,Image ID_
315,Traffic,https://www.johnsongarcialaw.com/wp-content/up...,Violated,Driving Distraction,Driving_Distraction_1,0000013,NaN,nan
316,Traffic,https://www.johnsongarcialaw.com/wp-content/up...,Not Applicable,Pedestrian Crossing,Driving_Distraction_1,0000013,NaN,nan
317,Traffic,https://www.johnsongarcialaw.com/wp-content/up...,Not Applicable,Road Condition,Driving_Distraction_1,0000013,NaN,nan
318,Traffic,https://www.johnsongarcialaw.com/wp-content/up...,Not Applicable,Traffic Rules,Driving_Distraction_1,0000013,NaN,nan
319,Traffic,https://www.johnsongarcialaw.com/wp-content/up...,Not Applicable,Vehicle Load,Driving_Distraction_1,0000013,NaN,nan
1780,Traffic,https://img.philkotse.com/temp/2024/07/27/inte...,Not Applicable,Driving Distraction,Traffic_Rules_0,0000034,NaN,nan
1781,Traffic,https://img.philkotse.com/temp/2024/07/27/inte...,Complied,Pedestrian Crossing,Traffic_Rules_0,0000034,NaN,nan
1782,Traffic,https://img.philkotse.com/temp/2024/07/27/inte...,Complied,Road Condition,Traffic_Rules_0,0000034,NaN,nan
1783,Traffic,https://img.philkotse.com/temp/2024/07/27/inte...,Complied,Traffic Rules,Traffic_Rules_0,0000034,NaN,nan
1784,Traffic,https://img.philkotse.com/temp/2024/07/27/inte...,Complied,Vehicle Load,Traffic_Rules_0,0000034,NaN,nan


In [261]:
df.loc[(df['Image ID'] != df['Image ID_']) & (~df['Image Path'].isna()), 'Note'] = 'Duplication'
df.loc[df['Image Path'].isna(), 'Note'] = 'Wrong Format (e.g. .webp)'


In [262]:
df['File'].value_counts()/5

File
Road_Condition_1         60.0
Driving_Distraction_1    58.0
Pedestrian_Crossing_1    54.0
Traffic_Rules_1          52.0
Driving_Distraction_0    51.0
Pedestrian_Crossing_0    51.0
Vehicle_Load_0           51.0
Vehicle_Load_1           50.0
Road_Condition_0         49.0
Traffic_Rules_0          47.0
Name: count, dtype: float64

Fix Label

In [263]:
total_distribution = df['Label'].value_counts()
total_distribution

Label
Not Applicable    1679
Complied           612
Violated           321
Compiled             2
Compled              1
Name: count, dtype: int64

In [264]:
df['Label'] = df['Label'].apply(lambda x: x.replace('Compiled', 'Complied').replace('Compled', 'Complied'))

In [266]:
final_df = df[df['Note'].isna()].copy()

In [270]:
total_distribution = final_df['Label'].value_counts()
total_distribution

Label
Not Applicable    1654
Complied           601
Violated           315
Name: count, dtype: int64

In [267]:
domain_rule_distribution = final_df.groupby(['Domain', 'Rule'])['Label'].value_counts().unstack(fill_value=0)
domain_rule_distribution

Label                        Complied  Not Applicable  Violated
Domain  Rule                                                   
Traffic Driving Distraction        55             401        58
        Pedestrian Crossing        58             401        55
        Road Condition            204             237        73
        Traffic Rules              98             338        78
        Vehicle Load              186             277        51

In [268]:
domain_rule_distribution['Total'] = domain_rule_distribution.sum(axis=1)
domain_rule_distribution = domain_rule_distribution.reset_index()


In [269]:
df.to_csv('data/hazard-project-annotation-cleaned.csv', index = False)